In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)

In [2]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import broadcast, col


In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.jars", "/opt/spark/jars/postgresql-42.6.0.jar") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.secret.key", "Projeto_Final") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [4]:
# ==========================================================
# DEFINIÇÃO DOS SCHEMAS
# ==========================================================
schema_previsao = StructType([
    StructField("regiao", StringType(), True),
    StructField("codigo_linha", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("hora_coleta", StringType(), True),
    StructField("codigo_parada", StringType(), True),
    StructField("nome_parada", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("prefixo_veiculo", StringType(), True),
    StructField("hora_prevista", StringType(), True),
    StructField("acessivel", BooleanType(), True)
])

schema_posicao = StructType([
    StructField("hr", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("id_linha", StringType(), True),
    StructField("terminal_origem", StringType(), True),
    StructField("terminal_destino", StringType(), True),
    StructField("sentido", IntegerType(), True),
    StructField("prefixo", StringType(), True),
    StructField("operando", StringType(), True),
    StructField("atualizacao", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("latitude", DoubleType(), True)
])

schema_gtfs = StructType([
    StructField("route_id", StringType(), True),
    StructField("stop_id", StringType(), True),
    StructField("stop_name", StringType(), True),
    StructField("stop_lat", DoubleType(), True),
    StructField("stop_lon", DoubleType(), True)
])


In [5]:
# -------------------------
# Leitura em streaming (Silver)
# -------------------------
previsao_stream = (
    spark.readStream
    .format("parquet")
    .schema(schema_previsao)
    .load("s3a://silver/previsao_chegada/*.parquet")
)

posicoes_stream = (
    spark.readStream
    .format("parquet")
    .schema(schema_posicao)
    .load("s3a://silver/posicoes/*.parquet")
)

gtfs_df = spark.read.parquet("s3a://silver/gtfs/*.parquet")

In [6]:
# ==========================================================
# Linhas monitoradas
# ==========================================================
linhas_monitoradas = {
    "zona_norte": [543, 614, 558, 2495, 709],
    "zona_sul": [1140, 59, 1977, 1318],
    "zona_leste": [2160, 1055, 2580, 1808],
    "zona_oeste": [689, 1376, 782, 472],
    "centro": [1523, 768, 2506, 1366]
}

linhas_filtradas = list(set(sum(linhas_monitoradas.values(), [])))


In [7]:
# -------------------------
# Função do job batch (join)
# -------------------------
def process_batch():
    ...
    postgres_url = "jdbc:postgresql://postgres:5432/sptrans_gold"
    postgres_props = {
        "user": "PROJETO_FINAL",
        "password": "PROJETO_FINAL",
        "driver": "org.postgresql.Driver"
    }

    final_tipado = (
        final_enriquecido
        .withColumn("timestamp", to_timestamp("timestamp"))
        .withColumn("hora_coleta", to_timestamp("hora_coleta"))
        .withColumn("hora_prevista", to_timestamp("hora_prevista"))
    )

    (
        final_tipado.write
        .mode("append")
        .option("batchsize", "5000")
        .option("truncate", "false")
        .jdbc(postgres_url, "public.previsao_posicoes_enriquecida", properties=postgres_props)
    )

    print("Job batch concluído e dados salvos no Postgres!")

    # Leitura mais recente das tabelas Silver
previsao_df = spark.read.parquet("s3a://silver/previsao_chegada/*.parquet")
posicoes_df = spark.read.parquet("s3a://silver/posicoes/*.parquet")

    # Filtrar linhas monitoradas
previsao_filtrada = previsao_df.filter(col("codigo_linha").cast("int").isin(linhas_filtradas))
posicoes_filtradas = posicoes_df.filter(col("id_linha").cast("int").isin(linhas_filtradas))

    # Selecionar colunas relevantes
posicoes_sel = posicoes_filtradas.select(
        col("prefixo"),
        col("route_id"),
        col("id_linha"),
        col("latitude").alias("lat_onibus"),
        col("longitude").alias("lng_onibus"),
        col("hr").alias("hora_onibus_pos"),
        col("operando").alias("operando_onibus"),
        col("terminal_origem"),
        col("terminal_destino")
    )

    # Join previsao + posicoes
previsao_enriquecida = previsao_filtrada.join(
        broadcast(posicoes_sel),
        previsao_filtrada["prefixo_veiculo"] == posicoes_sel["prefixo"],
        how="left"
    )

    # Join com GTFS (stop name, lat/lon)
gtfs_sel = gtfs_df.select(
        col("route_id").alias("gtfs_route_id"),
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    )

final_enriquecido = previsao_enriquecida.join(
        broadcast(gtfs_sel),
        (previsao_enriquecida["route_id"] == gtfs_sel["gtfs_route_id"]) &
        (previsao_enriquecida["codigo_parada"] == gtfs_sel["stop_id"]),
        how="left"
    ).select(
        "timestamp", "hora_coleta",
        "codigo_parada", "nome_parada",
        "latitude", "longitude",
        "prefixo_veiculo", "hora_prevista", "acessivel",
        "id_linha", "route_id",
        "lat_onibus", "lng_onibus", "hora_onibus_pos", "operando_onibus",
        col("stop_name").alias("stop_name_gtfs"),
        "stop_lat", "stop_lon", "terminal_origem", "terminal_destino"
    )


In [9]:
from pyspark.sql.functions import col, broadcast, to_timestamp

# ======================================================
# Escrita no Postgres (Camada Gold)
# ======================================================
postgres_url = "jdbc:postgresql://postgres:5432/sptrans_gold"
postgres_props = {
    "user": "PROJETO_FINAL",
    "password": "PROJETO_FINAL",
    "driver": "org.postgresql.Driver"
}

# Conversão opcional de timestamps
final_tipado = (
    final_enriquecido
    .withColumn("timestamp", to_timestamp("timestamp"))
    .withColumn("hora_coleta", to_timestamp("hora_coleta"))
    .withColumn("hora_prevista", to_timestamp("hora_prevista"))
)

(
    final_tipado.write
    .mode("append")  # ou "overwrite" se quiser recriar a tabela
    .option("batchsize", "5000")
    .option("truncate", "false")
    .jdbc(postgres_url, "public.previsao_posicoes_enriquecida", properties=postgres_props)
)

print("✅ Job batch concluído e dados salvos no Postgres!\n")


✅ Job batch concluído e dados salvos no Postgres!



In [11]:
# ==========================================================
#  LOOP DE EXECUÇÃO A CADA 3 MINUTOS
# ==========================================================
if __name__ == "__main__":
    while True:
        process_batch()
        print("Aguardando 3 minutos para próxima execução...\n")
        time.sleep(180)

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...

Job batch concluído e dados salvos no Postgres!
Aguardando 3 minutos para próxima execução...



KeyboardInterrupt: 